# EdNet 전처리

`02_build_tables`에서 만든 테이블에 연결만 해서 분석합니다. 재시작해도 테이블을 다시 만들지 않습니다.

## 0. DB 연결
`read_only=True`: 분석 중 실수로 테이블을 덮어쓰지 않도록 읽기 전용으로 엽니다.

In [4]:
import duckdb

con = duckdb.connect("../data/ednet.duckdb", read_only=True)

## 1. `user_id` 검증
빈 값이 없는지, 모두 `u숫자` 형식인지 확인합니다 → 이상 없음, 유저 784,309명

In [5]:
con.sql("""
SELECT
    COUNT(*) FILTER (WHERE user_id IS NULL) AS null_count,
    COUNT(*) FILTER (WHERE NOT regexp_matches(user_id, '^u[0-9]+$')) AS bad_format_count,
    COUNT(DISTINCT user_id) AS user_count
FROM kt1
""").df()

,null_count,bad_format_count,user_count
0,0,0,784309


## 2. 유저별 푼 문제 수 분포
중앙값은 10문제지만 평균은 약 102문제, 최대 11,466문제로 **소수의 헤비 유저에 크게 치우친 분포**입니다.

In [6]:
summary = con.sql("""
WITH per_user AS (
    SELECT
        user_id,
        COUNT(DISTINCT question_id) AS solved_count
    FROM kt1
    GROUP BY user_id
)
SELECT
    MIN(solved_count) AS min_solved,
    quantile_cont(solved_count, 0.25) AS q1,
    median(solved_count) AS median_solved,
    quantile_cont(solved_count, 0.75) AS q3,
    MAX(solved_count) AS max_solved,
    AVG(solved_count) AS avg_solved
FROM per_user
""").df()

summary

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,min_solved,q1,median_solved,q3,max_solved,avg_solved
0,1,7.0,10.0,37.0,11466,101.911324
